# MLP Function Approximation: 1D Consolidation Settlement [Exercise]

Every blank in this notebook is a line beginning `# TODO:`. The comment names the
symbols the cell must define, because later cells call them by name.

**Exercise:** [![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-F9AB00?style=flat-square&logo=googlecolab)](https://colab.research.google.com/github/kks32-courses/ai-geotech/blob/main/docs/03-mlp/03c-mlp-function-approximation-exercise.ipynb)
**Solution:** [![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-F9AB00?style=flat-square&logo=googlecolab)](https://colab.research.google.com/github/kks32-courses/ai-geotech/blob/main/docs/03-mlp/03c-mlp-function-approximation.ipynb)

In [ ]:
%pip install -q torch numpy matplotlib


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)
np.random.seed(42)


def seed_everything(seed=42):
    """Reseed torch before building a model, so any cell reruns to the same numbers."""
    torch.manual_seed(seed)
    np.random.seed(seed)


plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## Problem: 1D Consolidation Settlement

**Goal:** Predict settlement vs time from sparse field measurements.

We generate the data from an exponential model of settlement:
$$S(t) = S_{\text{final}}(1 - e^{-\alpha t})$$

- $S_{\text{final}} = 100$ mm (ultimate settlement)
- $\alpha = 0.5$ /year (settlement rate constant, units of 1/time)
- Sparse field data: 10 measurements over 10 years

This single exponential is not Terzaghi's one-dimensional consolidation solution. That
solution is a Fourier series,
$$U(T_v) = 1 - \sum_{m=0}^{\infty} \frac{2}{M^2} e^{-M^2 T_v},
\qquad M = \frac{(2m+1)\pi}{2},$$
whose first mode is $U \approx 1 - (8/\pi^2) e^{-\pi^2 T_v / 4}$. The prefactor $8/\pi^2$
matters. At early time the series rises like $\sqrt{t}$ while $1 - e^{-\alpha t} \approx
\alpha t$ is linear in $t$, and the first four data points below sit in that early-time
range. The single exponential is a reasonable approximation past about $U = 60\%$. We use
it here because it gives a smooth target with two parameters, not because it is the
consolidation solution.


In [ ]:
# True parameters
S_final = 100.0
alpha = 0.5

# Field measurements (sparse)
t_field = np.array([0, 0.5, 1, 1.5, 2, 3, 4, 6, 8, 10])
S_true = S_final * (1 - np.exp(-alpha * t_field))
noise_sigma = 1.5
S_measured = S_true + np.random.normal(0, noise_sigma, len(t_field))
# No clipping at zero. The true value at t = 0 is exactly 0, so clipping would remove
# half the noise on that point and bias it upward.

# Dense evaluation grid
t_dense = np.linspace(0, 10, 200)
S_dense = S_final * (1 - np.exp(-alpha * t_dense))

# Plot field data
plt.figure(figsize=(10, 5))
plt.scatter(t_field, S_measured, s=100, color='black', label='Field measurements', zorder=5)
plt.plot(t_dense, S_dense, 'k--', linewidth=2, label='True: $S(t) = 100(1-e^{-0.5t})$', alpha=0.7)
plt.axhline(S_final, color='gray', linestyle=':', linewidth=2, alpha=0.6)
plt.xlabel('Time (years)')
plt.ylabel('Settlement (mm)')
plt.title('Field Consolidation Measurements')
plt.legend()
plt.xlim(-0.5, 10.5)
plt.ylim(-5, 110)
plt.tight_layout()
plt.show()

print(f"Measurements: {len(t_field)} points")
print(f"Noise standard deviation: {noise_sigma} mm")
print(f"Variance floor a correct model cannot beat: {noise_sigma**2:.2f} mm\u00b2")
print("Challenge: Predict settlement at any time in [0, 10] years")

## Part 1: Single Neuron (Perceptron)

$$\hat{S} = w \cdot t + b$$

Can a linear model fit consolidation?

In [ ]:
# Convert to tensors
# TODO: t_train, S_train and t_eval as torch.FloatTensor of shape (-1, 1) from
#       t_field, S_measured and t_dense

# Linear model
# TODO: class LinearModel(nn.Module) wrapping a single nn.Linear(1, 1)
# TODO: seed_everything(); model_linear = LinearModel()
# TODO: optimizer = optim.Adam(model_linear.parameters(), lr=0.1)
# TODO: criterion = nn.MSELoss()          <- later cells use this name
# TODO: losses = []                       <- later cells plot this list
# TODO: run 1000 epochs, appending loss.item() to losses each epoch
# TODO: S_pred_linear = model_linear(t_eval).numpy() under torch.no_grad()
# TODO: print the final loss and the fitted slope and intercept


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Predictions
ax1.scatter(t_field, S_measured, s=100, color='black', label='Field data', zorder=5)
ax1.plot(t_dense, S_dense, 'k--', linewidth=2, label='True curve', alpha=0.7)
ax1.plot(t_dense, S_pred_linear, 'r-', linewidth=2, label='Linear model')
ax1.axhline(S_final, color='gray', linestyle=':', linewidth=2, alpha=0.6)
ax1.set_xlabel('Time (years)')
ax1.set_ylabel('Settlement (mm)')
ax1.set_title('Linear Model vs True Consolidation')
ax1.legend()
ax1.set_xlim(-0.5, 10.5)
ax1.set_ylim(-5, 110)

# Loss
ax2.plot(losses, 'r-', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss (MSE)')
ax2.set_title('Training Loss')
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

print("\n⚠️ Linear model cannot capture exponential decay!")

## Part 2: Adding Nonlinearity

Single layer with $H$ hidden units:
$$\hat{S}(t) = \sum_{j=1}^{H} w_{2,j}\, \sigma(w_{1,j} t + b_{1,j}) + b_2$$

where $\sigma = \tanh$. With $H = 1$ this is the scalar form
$\hat{S} = w_2 \sigma(w_1 t + b_1) + b_2$. The cell below uses $H = 10$, so $w_1$ and
$w_2$ are vectors of length 10 and $b_1$ is a vector of length 10.


In [ ]:
# TODO: class SingleLayerNN(nn.Module) with fc1 = Linear(1, n_hidden),
#       fc2 = Linear(n_hidden, 1) and torch.tanh between them.
#       Later cells construct it by this exact name.
# TODO: seed_everything(); model_tanh = SingleLayerNN(n_hidden=10)
# TODO: optimizer = optim.Adam(model_tanh.parameters(), lr=0.01)
# TODO: losses_tanh = []                  <- later cells plot this list
# TODO: run 2000 epochs, appending loss.item() to losses_tanh
# TODO: S_pred_tanh = model_tanh(t_eval).numpy() under torch.no_grad()
# TODO: print the final loss


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Predictions
ax1.scatter(t_field, S_measured, s=100, color='black', label='Field data', zorder=5)
ax1.plot(t_dense, S_dense, 'k--', linewidth=2, label='True curve', alpha=0.7)
ax1.plot(t_dense, S_pred_linear, 'r-', linewidth=2, label='Linear', alpha=0.5)
ax1.plot(t_dense, S_pred_tanh, 'b-', linewidth=2, label='Tanh (10 neurons)')
ax1.axhline(S_final, color='gray', linestyle=':', linewidth=2, alpha=0.6)
ax1.set_xlabel('Time (years)')
ax1.set_ylabel('Settlement (mm)')
ax1.set_title('Nonlinear Model: Much Better!')
ax1.legend()
ax1.set_xlim(-0.5, 10.5)
ax1.set_ylim(-5, 110)

# Loss comparison
ax2.plot(losses, 'r-', linewidth=2, label='Linear', alpha=0.7)
ax2.plot(losses_tanh, 'b-', linewidth=2, label='Tanh')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss (MSE)')
ax2.set_title('Training Loss Comparison')
ax2.set_yscale('log')
ax2.legend()

plt.tight_layout()
plt.show()

## Part 3: Effect of Network Width

How many neurons do we need?

In [ ]:
def train_model(n_hidden, epochs=2000, seed=42):
    """Return (S_pred, training loss, error against the true curve, parameter count)."""
    seed_everything(seed)
    # TODO: build SingleLayerNN(n_hidden), an Adam optimizer at lr=0.01, and run
    #       `epochs` steps on t_train and S_train
    # TODO: S_pred = model(t_eval).numpy() under torch.no_grad()
    # TODO: err_true = float(np.mean((S_pred - S_dense.reshape(-1, 1))**2))
    # TODO: n_params = sum(p.numel() for p in model.parameters())
    # TODO: return S_pred, loss.item(), err_true, n_params
    raise NotImplementedError("Fill in train_model, then delete this line")


# Try different widths
widths = [1, 5, 10, 20, 50]
results = {}

noise_floor = noise_sigma**2
print(f"Noise variance on the training targets: {noise_floor:.2f} mm\u00b2")
print(f"{'Width':>6} {'Params':>7} {'Train loss':>12} {'Err vs true':>12}")
for w in widths:
    S_pred, final_loss, err_true, n_params = train_model(w)
    results[w] = {'pred': S_pred, 'loss': final_loss, 'err_true': err_true, 'params': n_params}
    print(f"{w:>6} {n_params:>7} {final_loss:>12.3f} {err_true:>12.3f}")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for idx, w in enumerate(widths):
    ax = axes[idx]
    ax.scatter(t_field, S_measured, s=80, color='black', label='Data', zorder=5)
    ax.plot(t_dense, S_dense, 'k--', linewidth=2, label='True', alpha=0.7)
    ax.plot(t_dense, results[w]['pred'], 'b-', linewidth=2, label='MLP')
    ax.axhline(S_final, color='gray', linestyle=':', linewidth=1, alpha=0.6)
    ax.set_title(f'{w} neurons: train {results[w]["loss"]:.2f}, vs true {results[w]["err_true"]:.2f}')
    ax.set_xlabel('Time (years)')
    ax.set_ylabel('Settlement (mm)')
    ax.legend(fontsize=8)
    ax.set_xlim(-0.5, 10.5)
    ax.set_ylim(-5, 110)

# Remove extra subplot
axes[-1].axis('off')

plt.tight_layout()
plt.show()

best_w = min(widths, key=lambda w: results[w]['err_true'])
under_floor = [w for w in widths if results[w]['loss'] < noise_floor]
print(f"\nNoise variance on the targets: {noise_floor:.2f} mm\u00b2")
print(f"Widths whose training loss fell below it (fitting the noise): {under_floor}")
print(f"Lowest error against the true curve: width {best_w} at {results[best_w]['err_true']:.3f} mm\u00b2")

## Part 4: Deep Network (2 Hidden Layers)

$$\hat{S} = w_3 \cdot \sigma(w_2 \cdot \sigma(w_1 \cdot t + b_1) + b_2) + b_3$$

In [ ]:
# Deep Neural Network
# TODO: class DeepNN(nn.Module) with fc1 = Linear(1, n_hidden),
#       fc2 = Linear(n_hidden, n_hidden), fc3 = Linear(n_hidden, 1) and torch.tanh
#       after fc1 and fc2. Cell 19 constructs DeepNN by this exact name.
# TODO: seed_everything(); model_deep = DeepNN(n_hidden=16)
# TODO: optimizer = optim.Adam(model_deep.parameters(), lr=0.01)
# TODO: losses_deep = []                  <- later cells plot this list
# TODO: run 2000 epochs, appending loss.item() to losses_deep
# TODO: S_pred_deep = model_deep(t_eval).numpy() under torch.no_grad()
# TODO: print the final loss, then the parameter counts of model_deep and model_tanh,
#       and note that depth is confounded with size in this comparison


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Predictions
ax1.scatter(t_field, S_measured, s=100, color='black', label='Field data', zorder=5)
ax1.plot(t_dense, S_dense, 'k--', linewidth=2, label='True curve', alpha=0.7)
ax1.plot(t_dense, S_pred_tanh, 'b-', linewidth=2, label='1-layer (10 neurons)', alpha=0.7)
ax1.plot(t_dense, S_pred_deep, 'g-', linewidth=2, label='2-layer (16-16)')
ax1.axhline(S_final, color='gray', linestyle=':', linewidth=2, alpha=0.6)
ax1.set_xlabel('Time (years)')
ax1.set_ylabel('Settlement (mm)')
ax1.set_title('Shallow vs Deep Network')
ax1.legend()
ax1.set_xlim(-0.5, 10.5)
ax1.set_ylim(-5, 110)

# Loss
ax2.plot(losses_tanh, 'b-', linewidth=2, label='1-layer', alpha=0.7)
ax2.plot(losses_deep, 'g-', linewidth=2, label='2-layer')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss (MSE)')
ax2.set_title('Training Loss')
ax2.set_yscale('log')
ax2.legend()

plt.tight_layout()
plt.show()

## Part 5: Optimizer Comparison

SGD vs Momentum vs Adam

In [ ]:
def train_optimizer(opt_name, lr=0.01, epochs=2000, seed=42):
    seed_everything(seed)
    model = DeepNN(n_hidden=16)

    if opt_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr)
    elif opt_name == 'Momentum':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif opt_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)

    losses = []
    for epoch in range(epochs):
        optimizer.zero_grad()
        S_pred = model(t_train)
        loss = criterion(S_pred, S_train)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    with torch.no_grad():
        S_pred = model(t_eval).numpy()

    return losses, S_pred

optimizers = ['SGD', 'Momentum', 'Adam']
opt_results = {}

# All three run at the same lr = 0.01, which is not a fair comparison. A rate that
# suits plain gradient descent can be far too large once momentum multiplies it.
for opt in optimizers:
    print(f"Training with {opt}...")
    losses, S_pred = train_optimizer(opt)
    opt_results[opt] = {'losses': losses, 'pred': S_pred}
    print(f"  Final loss: {losses[-1]:.4f}, best loss seen: {min(losses):.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Predictions
ax1.scatter(t_field, S_measured, s=100, color='black', label='Field data', zorder=5)
ax1.plot(t_dense, S_dense, 'k--', linewidth=2, label='True curve', alpha=0.7)
colors = {'SGD': 'red', 'Momentum': 'blue', 'Adam': 'green'}
for opt in optimizers:
    ax1.plot(t_dense, opt_results[opt]['pred'], color=colors[opt], linewidth=2, label=opt, alpha=0.8)
ax1.axhline(S_final, color='gray', linestyle=':', linewidth=2, alpha=0.6)
ax1.set_xlabel('Time (years)')
ax1.set_ylabel('Settlement (mm)')
ax1.set_title('Optimizer Comparison')
ax1.legend()
ax1.set_xlim(-0.5, 10.5)
ax1.set_ylim(-5, 110)

# Loss curves
for opt in optimizers:
    ax2.plot(opt_results[opt]['losses'], color=colors[opt], linewidth=2, label=opt, alpha=0.7)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss (MSE)')
ax2.set_title('Training Loss: Convergence Speed')
ax2.set_yscale('log')
ax2.legend()

plt.tight_layout()
plt.show()

ranked = sorted(optimizers, key=lambda o: opt_results[o]['losses'][-1])
print("\nFinal loss, lowest first, all at lr = 0.01:")
for o in ranked:
    print(f"  {o:<9} {opt_results[o]['losses'][-1]:.4f}")
target = 10.0
for o in optimizers:
    hit = next((i for i, l in enumerate(opt_results[o]['losses']) if l < target), None)
    print(f"  {o:<9} reached loss {target} at epoch {hit}" if hit is not None
          else f"  {o:<9} never reached loss {target}")
print("\nA single shared learning rate is not a fair comparison between optimizers.")

## Part 6: Activation Function Comparison

ReLU vs Tanh vs Sigmoid

In [ ]:
class MLPActivation(nn.Module):
    def __init__(self, activation='tanh', n_hidden=16):
        super().__init__()
        self.fc1 = nn.Linear(1, n_hidden)
        self.fc2 = nn.Linear(n_hidden, n_hidden)
        self.fc3 = nn.Linear(n_hidden, 1)

        if activation == 'relu':
            self.act = nn.ReLU()
        elif activation == 'tanh':
            self.act = nn.Tanh()
        elif activation == 'sigmoid':
            self.act = nn.Sigmoid()

    def forward(self, t):
        t = self.act(self.fc1(t))
        t = self.act(self.fc2(t))
        return self.fc3(t)

def train_activation(activation, epochs=3000, seed=42):
    seed_everything(seed)
    model = MLPActivation(activation=activation, n_hidden=32)
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    losses = []
    for epoch in range(epochs):
        optimizer.zero_grad()
        S_pred = model(t_train)
        loss = criterion(S_pred, S_train)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    with torch.no_grad():
        S_pred = model(t_eval).numpy()

    # Mean squared difference from the true curve on [0, 10], the same interval the
    # model was fit on. There is no test set in this notebook.
    err_true = np.mean((S_pred - S_dense.reshape(-1, 1))**2)
    return losses, S_pred, err_true

activations = ['relu', 'tanh', 'sigmoid']
act_results = {}

for act in activations:
    print(f"Training with {act.upper()}...")
    losses, S_pred, err_true = train_activation(act)
    act_results[act] = {'losses': losses, 'pred': S_pred, 'err_true': err_true}
    print(f"  Error against the true curve: {err_true:.2f} mm²")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Predictions
ax1.scatter(t_field, S_measured, s=120, color='black', label='Field data', zorder=5)
ax1.plot(t_dense, S_dense, 'k--', linewidth=2, label='True curve', alpha=0.7)
colors_act = {'relu': 'red', 'tanh': 'blue', 'sigmoid': 'green'}
for act in activations:
    ax1.plot(t_dense, act_results[act]['pred'], color=colors_act[act],
             linewidth=2.5, label=f'{act.upper()} (err vs true={act_results[act]["err_true"]:.1f})', alpha=0.8)
ax1.axhline(S_final, color='gray', linestyle=':', linewidth=2, alpha=0.6)
ax1.set_xlabel('Time (years)')
ax1.set_ylabel('Settlement (mm)')
ax1.set_title('Activation Function Comparison')
ax1.legend(loc='lower right')
ax1.set_xlim(-0.5, 10.5)
ax1.set_ylim(-5, 110)

# Loss curves
for act in activations:
    ax2.plot(act_results[act]['losses'], color=colors_act[act], linewidth=2,
             label=f'{act.upper()}', alpha=0.7)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss (MSE)')
ax2.set_title('Training Loss')
ax2.set_yscale('log')
ax2.legend()

plt.tight_layout()
plt.show()

### Boundary Behavior Analysis

In [ ]:
print(f"True settlement at t=10 years: {S_dense[-1]:.2f} mm")
print(f"Physical limit S_final: {S_final:.2f} mm\n")
print(f"{'Activation':<12} {'Prediction':<12} {'Error':<10} {'Over S_final?':<14}")
print("-" * 55)

for act in activations:
    pred_10 = act_results[act]['pred'][-1][0]
    error = pred_10 - S_dense[-1]
    over = 'yes' if pred_10 > S_final else 'no'
    print(f"{act.upper():<12} {pred_10:>10.2f} mm {error:>9.2f} mm {over:>10}")

n_over = sum(act_results[a]['pred'][-1][0] > S_final for a in activations)
print(f"\n{n_over} of {len(activations)} predictions exceed S_final = {S_final:.0f} mm.")
print("The output layer fc3 is linear, so no hidden activation bounds the prediction.")
print("A bound would need S_final * sigmoid(...) at the output.")


## Summary

**Key findings:**

1. **Linear models fail** for nonlinear consolidation.
2. **Width**: the error against the true curve falls from 8.33 at 10 neurons to 1.42
   at 20 and 1.24 at 50. Past 20 neurons the gain is small. Training loss keeps
   falling further, to 0.29 at 20 neurons and 0.19 at 50, both below the 2.25 mm²
   noise variance. A model cannot beat the noise variance without fitting the noise.
   Read the "err vs true" column, not the training loss.
3. **Depth**: the two-layer network reaches a lower loss than the one-layer network,
   but it also has ten times the parameters. Depth is confounded with size in this
   comparison. Matching the parameter count would separate the two.
4. **Optimizers**: at the shared learning rate of 0.01 the three optimizers do not
   rank the way the textbook ordering suggests. The cell above prints the ranking from
   the run. Momentum multiplies the effective step, so a rate that suits plain gradient
   descent can make the momentum run diverge. A single shared rate is not a fair test.
5. **Activations**: tanh reaches the lowest error against the true curve (2.31 mm²),
   sigmoid is next (2.93) and ReLU last (3.98). At $t = 10$ years ReLU and tanh both
   predict above $S_{\text{final}} = 100$ mm. The output layer is linear, so the
   choice of hidden activation cannot enforce $0 \leq S \leq S_{\text{final}}$.
   Applying $S_{\text{final}} \cdot \sigma(\cdot)$ at the output would.

**Practical starting point** for settlement prediction from sparse field data:
- Architecture: 2 layers, 16-32 neurons per layer
- Activation: Tanh
- Optimizer: Adam with lr=0.01
- Training: 2000-3000 epochs

Every number in this notebook comes from 10 noisy points. See
`03d-mlp-extrapolation.ipynb` for what these models do outside the fitted interval.
